# 02b — Build structured intervals

Run and inspect each stage. Delivery 2 structures net-meter evidence; it does not assess inverter conformance.

In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p/'src'/'ausgrid_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the ausgrid_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT/'src'))
from ausgrid_analysis.config import load_config
from ausgrid_analysis.db import connect, site_phase_profile_path, site_profile_path, structured_phase_output_path, structured_site_output_path
from ausgrid_analysis.delivery2_profiles import build_site_profiles
from ausgrid_analysis.delivery2_structured import build_structured_phase, build_structured_site
from ausgrid_analysis.delivery2_validation import validate_delivery2
from ausgrid_analysis.schemas import sql_string
pd.set_option('display.max_columns', 100); plt.style.use('seaborn-v0_8-whitegrid')

## Methodology contract

Before running, review `docs/METHODOLOGY_GATES.md`. Load/PV decomposition, battery separation, sign verification, uncurtailed PV and curve comparison are deliberately deferred. Candidate inverter phases receive a confidence; low/unknown sites are not silently promoted.

In [ ]:
CONFIG_PATH = PROJECT_ROOT/'analysis.toml'
SAMPLE_MONTH = '2025-04'
SAMPLE_SITE_BUCKET = 0
OVERWRITE_SAMPLE = False
config = load_config(CONFIG_PATH, check_inputs=False)
scope = config.scope(SAMPLE_MONTH, SAMPLE_SITE_BUCKET)
print('Sample scope:', scope.label)
print('Measurement basis: net meter; voltage location: revenue meter')

## Stage 1 — site/phase profiles and candidate DER-phase mapping

In [ ]:
if OVERWRITE_SAMPLE or not (site_phase_profile_path(config, scope).is_file() and site_profile_path(config, scope).is_file()):
    profile_summary = build_site_profiles(config, scope, overwrite=OVERWRITE_SAMPLE)
    display(pd.DataFrame([profile_summary]).T.rename(columns={0:'value'}))
else: print('Reusing existing sample profiles. Set OVERWRITE_SAMPLE=True to rebuild.')
con = connect(config)
phase_profiles = con.execute(f'SELECT * FROM read_parquet({sql_string(site_phase_profile_path(config, scope))}) ORDER BY serial, phase').fetchdf()
site_profiles = con.execute(f'SELECT * FROM read_parquet({sql_string(site_profile_path(config, scope))}) ORDER BY serial').fetchdf()
display(site_profiles.groupby(['analysis_cohort','phase_mapping_confidence'], dropna=False).size().rename('n_sites').reset_index())
display(site_profiles[['serial','analysis_cohort','has_battery','install_phase_count','power_available_phases','inferred_der_phases','phase_mapping_method','phase_mapping_confidence','delivery2_primary_cohort']].head(30))
display(phase_profiles.groupby('phase').agg(n_pairs=('serial','size'), power_missing=('power_measurement_available',lambda x:(~x).sum()), zero_voltage=('n_voltage_at_or_below_zero','sum'), median_signature=('solar_signature_w','median')))
assert site_profiles.serial.is_unique
assert not site_profiles.formal_inverter_conformance_assessable.any()
print('Stage 1 structural gate passed. Review low/unknown mappings before continuing.')

## Stage 2 — structured phase intervals

In [ ]:
phase_out = structured_phase_output_path(config, scope)
if OVERWRITE_SAMPLE or not phase_out.is_dir():
    display(pd.DataFrame([build_structured_phase(config, scope, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0:'value'}))
else: print('Reusing existing structured phase sample.')
phase_glob = str(phase_out/'**'/'*.parquet')
phase_check = con.execute(f'''SELECT count(*) n_rows, count(DISTINCT serial) n_sites,
 count_if(is_inferred_der_phase) inferred_der_phase_rows,
 count_if(NOT voltage_valid_for_analysis) invalid_voltage_rows,
 count_if(NOT power_measurement_available) missing_power_rows,
 count_if(utc_offset_minutes=600) aest_rows, count_if(utc_offset_minutes=660) aedt_rows
 FROM read_parquet({sql_string(phase_glob)}, hive_partitioning=true)''').fetchdf()
display(phase_check)
display(con.execute(f'SELECT * FROM read_parquet({sql_string(phase_glob)}, hive_partitioning=true) ORDER BY timestamp_utc, serial, phase LIMIT 20').fetchdf())

## Stage 3 — structured site intervals and safe aggregation

In [ ]:
site_out = structured_site_output_path(config, scope)
if OVERWRITE_SAMPLE or not site_out.is_dir():
    display(pd.DataFrame([build_structured_site(config, scope, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0:'value'}))
else: print('Reusing existing structured site sample.')
site_glob = str(site_out/'**'/'*.parquet')
site_preview = con.execute(f'''SELECT * FROM read_parquet({sql_string(site_glob)}, hive_partitioning=true)
 ORDER BY timestamp_utc, serial LIMIT 1000''').fetchdf()
display(site_preview.head(20))
display(site_preview.groupby(['analysis_cohort','has_battery','phase_mapping_confidence'], dropna=False).agg(rows=('serial','size'), complete=('der_phase_power_complete','sum')).reset_index())
assert site_preview.loc[~site_preview.der_phase_power_complete, 'p_export_der_phase_net_complete_w'].isna().all()

## Sample validation gate

In [ ]:
validation = validate_delivery2(config, scope)
display(pd.DataFrame([{k:v for k,v in validation.items() if k not in {'monthly_coverage','methodology_state','failures'}}]).T.rename(columns={0:'value'}))
display(pd.DataFrame(validation['monthly_coverage']))
display(pd.DataFrame([validation['methodology_state']]).T.rename(columns={0:'state'}))
assert validation['status']=='pass', validation['failures']
print('Delivery 2 sample passed. Interpret mappings scientifically before unlocking full build.')

## Full dataset build — locked

This can create another large phase-level dataset. Run only after reviewing the complete sample.

In [ ]:
FULL_RUN_CONFIRMATION = ''  # Change to: RUN DELIVERY 2 FULL
OVERWRITE_FULL = False
assert FULL_RUN_CONFIRMATION == 'RUN DELIVERY 2 FULL', 'Full run remains locked.'
full_scope = config.scope(None, None)
print('Full Delivery 2 unlocked:', full_scope.label)

In [ ]:
display(pd.DataFrame([build_site_profiles(config, full_scope, overwrite=OVERWRITE_FULL)]).T.rename(columns={0:'value'}))
display(pd.DataFrame([build_structured_phase(config, full_scope, overwrite=OVERWRITE_FULL)]).T.rename(columns={0:'value'}))
display(pd.DataFrame([build_structured_site(config, full_scope, overwrite=OVERWRITE_FULL)]).T.rename(columns={0:'value'}))
full_validation = validate_delivery2(config, full_scope)
display(pd.DataFrame([{k:v for k,v in full_validation.items() if k!='monthly_coverage'}]).T.rename(columns={0:'value'}))
display(pd.DataFrame(full_validation['monthly_coverage']))
assert full_validation['status']=='pass', full_validation['failures']
print('Delivery 2 full build passed.')
con.close()